# XuNet v4: shared training notebook

Experiment values belong in the first code cell. Machine values come from the
PC/G4 profile; a one-off `run_notebook.py --set key=value` is recorded separately.
The base notebook keeps defaults; derived notebooks record experiment values in their top cell.

`cvdl/` owns data, models, training, evaluation and diagnostics. The frozen
before-reference is `archive/05_baseline_kv_xunet_v4_pre-cvdl.ipynb`.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "cvdl").is_dir())
sys.path.insert(0, str(ROOT))
from cvdl.settings import load

settings = load(
    post_fit_eval="none",
    run_name='xunet_v4_clip3_nonorm_lr0.0003_warm1_drops5x0.3-20x0.1_k1to4_e30_wide-w4_r2',
    model='xunet_wide',
    width=4,
    norm='none',
    clip_t=3.0,
    lr=0.0003,
    warmup_epochs=1,
    grad_clip=1.0,
    lr_drops=((5, 0.3), (20, 0.1)),
    epochs=30,
    train_yaml='configs/train-k1to4.yaml',
    data='otf',
    source='decode',
    batch_size=32,
    extra_tests='test-bpp1',
)
# Example experiment: settings = load(model="xunet_wide", width=4, lr=3e-4, norm="none")
settings.apply_runtime_environment()

## Data and machine setup

Keep the standard test unchanged. QUICK limits normal evaluation and extra
panels. Data identities and settings are saved beside the checkpoints.

In [ ]:
import torch
from cvdl import runtime, data, evaluation, diagnostics
from cvdl.lit import SteganalysisModule
from cvdl.metrics import metrics
from lsb_torch import FixedPairs

runtime.configure(settings)
RUN_NAME, QUICK, V3_DATA = settings.run_name, settings.quick, settings.data
EPOCHS, BATCH_SIZE, COMPILE_MODE = settings.epochs, settings.batch_size, settings.compile
DEVICE = torch.device(settings.device)
otf_data, tables = runtime.create_data(settings)
print(f"Run: {RUN_NAME}; device={DEVICE}; epochs={EPOCHS}; batch={BATCH_SIZE}; QUICK={QUICK}")

In [ ]:
loaders, clean_batch, stego_batch = runtime.create_loaders(settings, otf_data, tables)
train_loader, val_loader = loaders.train, loaders.val
test_loader, train_eval_loader = loaders.test, loaders.train_eval
print(f"Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

## Model checks and training

The two small model checks retain the original random-number sequence and
verify that the KV filter stays frozen. Train with one best checkpoint and a
fresh `last.ckpt` after every epoch. Resume uses that final checkpoint.

CSV/TensorBoard logs, learning-rate traces and training curves stay in the run
folder. Use `tensorboard --logdir runs` (or the G4 output root) to watch training.

In [ ]:
runtime.model_checks(settings, clean_batch, stego_batch)

In [ ]:
bundle = runtime.create_training(settings, otf_data, loaders)
trainer, lightning_model = bundle.trainer, bundle.lightning_model
checkpoint_dir, checkpoint_callback = bundle.checkpoint_dir, bundle.checkpoint_callback
assert trainer.checkpoint_callbacks == [checkpoint_callback]

In [ ]:
trainer.fit(lightning_model, train_loader, val_loader, ckpt_path="last")
best_checkpoint = runtime.finish_training(settings, bundle, loaders)

## Predictions and results

Reload the best eager model for float32 evaluation. Save paired predictions,
print P_E/AUC summaries, run selected extra panels, and write the per-setting
CSV plus four breakdown heatmaps. Batch checks use copies of the best network.

In [ ]:
if settings.post_fit_eval == "full":
    best_model = SteganalysisModule.load_from_checkpoint(best_checkpoint, map_location=DEVICE, weights_only=False)
    best_model.to(DEVICE)
    train_predictions = evaluation.predict_pairs(best_model, train_eval_loader, device=DEVICE, data_mode=V3_DATA)
    val_predictions = evaluation.predict_pairs(best_model, val_loader, device=DEVICE, data_mode=V3_DATA)
    predictions = evaluation.predict_pairs(best_model, test_loader, device=DEVICE, data_mode=V3_DATA)
    for name, frame in (("train", train_predictions), ("val", val_predictions), ("test", predictions)):
        print(name, metrics(frame))
    print(evaluation.per_setting_table(predictions, data_mode=V3_DATA).to_string(index=False))
    predictions_path = evaluation.save_predictions(predictions, checkpoint_dir / "test_predictions.csv")

In [ ]:
if settings.post_fit_eval == "full":
    extra_results = evaluation.extra_panels(best_checkpoint, root=ROOT, output=checkpoint_dir,
        panels=settings.extra_tests, workers=settings.eval_workers, device=DEVICE, quick=QUICK,
        data_mode=V3_DATA, manifest_root=settings.manifest_root,
        source_panels=getattr(otf_data, "extra_pairs", ()))
    for name, (info, table) in extra_results.items():
        print(f"EXTRA {name}: {info['pairs']} pairs")
        print(table.to_string(index=False))

In [ ]:
if settings.post_fit_eval == "full":
    summary, per_setting, cell_table, heatmap_paths = evaluation.breakdown(predictions_path,
        labels_path=settings.labels_path if V3_DATA == "png" else None, show=True)
    print("TEST SUMMARY:", summary)
    print(per_setting.to_string(index=False))
    assert len(heatmap_paths) == 4
    print("OUTPUT CHECK: predictions, per-setting CSV and four heatmaps written")

In [ ]:
if settings.post_fit_eval == "full":
    diagnostic_results = {}
    for split_name in ("train", "val"):
        if V3_DATA == "otf":
            base = otf_data.fixed_train if split_name == "train" else FixedPairs(otf_data.corpus, "val", settings.eval_profile)
        else:
            base = data.PairDataset(tables.pair_splits.loc[tables.pair_splits["split"] == split_name], settings.data_root)
        selected = diagnostics.diagnostic_pairs(base, quick=QUICK, data_mode=V3_DATA,
            workers=settings.diag_workers, data_root=settings.data_root)
        if otf_data is not None:
            otf_data.record.setdefault("diagnostic_pair_ids", {})[split_name] = selected.frame["cover_id"].tolist()
        diagnostic_results[split_name] = diagnostics.batch_dependence(best_model.net, selected,
            device=DEVICE, norm=settings.norm, split_name=split_name, quick=QUICK,
            loader_workers=6 if V3_DATA == "png" else 0)
        print(f"CHECK {split_name}:", diagnostic_results[split_name])
    if otf_data is not None:
        otf_data.save_record(checkpoint_dir / "data_run_settings.json")
    settings.write(checkpoint_dir)
    print(f"Saved run: {checkpoint_dir}")